# 03 — Local graph methods: LLE, Laplacian eigenmaps (Jia)

Owner: **Jia**. Only Jia edits this notebook.

Template: load the shared data, fit your method, score it with the shared
metrics, compare with the PCA baseline. Name your outputs `03_*` in
`results/` and `figures/`.

## How to use the data (read first)

Full description: README, section **Data setup**.

1. **Splits.** `data.make_splits()` returns the 5 fixed splits; each has three disjoint index arrays `sp.tr`, `sp.val`, `sp.te` of 2000 images. The indices are stored in `splits/`, so everyone uses exactly the same images. Never pass other `n` or `seeds`.
2. **Tune on every split, both tracks:**
   - **Track A** (methods with a map, i.e. a `transform` for new points): fit on `X[sp.tr]`, map `X[sp.val]`, score with the out-of-sample $R_{NX}$ → `J_A_val`.
   - **Track B** (all methods): fit on `X[sp.val]`, score in-sample on the same points → `J_B_val`.
3. **Freeze** one configuration per (method, $q$, track): the best score **averaged over the 5 splits**, with `selection.select_frozen` and `selection.save_frozen` → `results/frozen_configs.csv`. Do not pick a different configuration per split.
4. **Report** with the frozen configuration on every split: Track A fit on `tr`, map `te`; Track B fit on `te`. `X[sp.te]` is for reporting and analysis only, never for choosing.
5. **Never** use the official test images (rows 60,000–69,999). `data.final_holdout_chunks` stays locked until everyone has frozen; the final run is done once, together.
6. Random seeds (t-SNE, UMAP, AE) are not tuned: keep the split fixed, vary the seed, report the spread.

Template for steps 2–3 below; replace the example method and grid with yours. Notebook `02_yifei_spectral` is a complete example.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))          # make nldr importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nldr import data, metrics

RESULTS, FIGURES = ROOT / "results", ROOT / "figures"
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)

def save(fig, name):
    """Save a figure as figures/<name>.png and .pdf."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=200, bbox_inches="tight")

def plot_rnx(curves, ax, title=None):
    """R_NX(K) against log K. curves: label -> (rnx, J)."""
    for label, (rnx, J) in curves.items():
        ax.semilogx(np.arange(1, len(rnx) + 1), rnx, label=f"{label} (J = {J:.3f})")
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set(xlabel="$K$ (log scale)", ylabel="$R_{NX}(K)$", title=title)
    ax.legend(fontsize=8)

In [ ]:
X, y = data.load_mnist()
s = data.make_splits()[0]               # quick check on split 0 only; protocol at the end
Xte, yte = X[s.te], y[s.te]
rho_te = metrics.rank_matrix(Xte)       # input ranks, reuse for every embedding
baseline = pd.read_csv(RESULTS / "baseline_pca_split0.csv", index_col="q")  # from 01_baselines
baseline

## Your method

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding
Y = LocallyLinearEmbedding(n_neighbors=10, n_components=2).fit_transform(Xte)

In [ ]:
res = metrics.evaluate(Xte, Y, yte, rho=rho_te)
rnx = res.pop("rnx")
print(f"J = {res['J']:.4f}   (PCA, q=2: {baseline.loc[2, 'J']:.4f})")

fig, ax = plt.subplots(figsize=(6, 4))
plot_rnx({"my method": (rnx, res["J"])}, ax, title="split 0 (test sample)")
save(fig, "03_example_rnx"); plt.show()

## Protocol: all splits, both tracks, freeze

The cells above are a quick check on split 0. For the results, use the protocol below (see "How to use the data").

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding
# Steps 2-3: score every configuration on every split, then freeze.
# Replace fit_B / fit_A and GRID with your method. LLE has a transform (Track A); for Laplacian eigenmaps use the Nystrom extension.
from nldr import selection

Q = 2                                            # also run q = 10 for the tables
GRID = [{"n_neighbors": k} for k in (5, 10, 20, 50)]

def label(params):
    return f"k={params['n_neighbors']}"

def fit_B(Z, params):
    """Track B: embedding of the points Z themselves, shape (n, Q)."""
    return LocallyLinearEmbedding(n_components=Q, random_state=0, **params).fit_transform(Z)

def fit_A(Z, params):
    """Track A: (embedding of Z, function mapping new points). None if no map."""
    m = LocallyLinearEmbedding(n_components=Q, random_state=0, **params).fit(Z)
    return m.embedding_, m.transform

splits = data.make_splits()
rows = []
for sp in splits:
    Xtr, Xval = X[sp.tr], X[sp.val]
    rho_val = metrics.rank_matrix(Xval)
    for params in GRID:
        row = {"split": sp.seed, "method": "lle", "config": label(params), "q": Q}
        Yb = fit_B(Xval, params)
        row["J_B_val"] = metrics.auc_log(metrics.rnx_curve(rho_val, metrics.rank_matrix(Yb)))
        fa = fit_A(Xtr, params)
        if fa is not None:
            Ytr, transform = fa
            row["J_A_val"] = metrics.auc_log(metrics.rnx_curve_oos(Xtr, Ytr, Xval, transform(Xval)))
        rows.append(row)
runs = pd.DataFrame(rows)
runs.round(4).to_csv(RESULTS / "03_all_configs.csv", index=False)

PARAMS = {("lle", label(p)): p for p in GRID}
frozen = [selection.select_frozen(runs, "J_B_val", "B", params=PARAMS)]
if "J_A_val" in runs:
    frozen.append(selection.select_frozen(runs, "J_A_val", "A", params=PARAMS))
frozen = pd.concat(frozen, ignore_index=True)
selection.save_frozen(frozen)                    # writes only your rows of results/frozen_configs.csv
frozen.drop(columns="params")
